# 03 - Training/validation split of ESM25

ESM25 is used for training and validation only; the test set will be made of later ESM records (events after the last
ESM25 event and not present in ESM25).

The split is grouped by event and stratified by label: all records of an event fall in the same set, so the validation
set contains only unseen events, as the test set will. Stations are largely shared between the sets, as they will be with
the test set: the results must be read in this light (or evaluated separately on stations not seen in training).

Parameters in the `split` section of `configs/default.yaml`. The split is saved once and reused by every experiment.

## 0. Setup

In [7]:
from pathlib import Path

import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import ORIGIN_TIME_COLUMN, load_esm25_metadata
from wqu.data.splitting import (
    SPLIT_COLUMN,
    assign_split,
    save_split,
    shared_events,
    split_counts,
    station_overlap,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
split = config.split
label_names = {subset.label: subset.name for subset in config.data.subsets}
split.tables_dir.mkdir(parents=True, exist_ok=True)

_, metadata = load_esm25_metadata(config)
print(f"Records: {len(metadata):,}; events: {metadata[columns.event_id].nunique():,}; "
      f"stations: {metadata[columns.station_id].nunique():,}")
print(f"Validation fraction {split.validation_fraction:g} -> {split.n_folds} folds, the first is the validation set; "
      f"seed {split.random_seed}")

Records: 100,249; events: 6,784; stations: 4,173
Validation fraction 0.2 -> 5 folds, the first is the validation set; seed 42


## 1. Assign and save the split

In [8]:
split_table = assign_split(metadata, columns, split)
split_path = save_split(split_table, split.output_file)
print(f"Split saved in {split_path}")
counts = split_counts(split_table, columns)
counts.to_csv(split.tables_dir / "split_counts.csv")
counts.round(3)

Split saved in /Users/giulianaparadiso/Desktop/Projects/waveform-quality-uncertainty/data/splits/esm25_train_validation.csv


,records,record_share,bad_rate,events,stations
split,,,,,
train,80193,0.8,0.359,5414,3627
validation,20056,0.2,0.359,1370,2850


## 2. Checks

- no event in both sets;
- how much the validation stations are known from training;
- class balance and main physical variables in the two sets (they should be similar).

In [9]:
n_shared_events = shared_events(split_table, columns)
if n_shared_events:
    raise RuntimeError(f"{n_shared_events} events are in both sets.")
print("Events shared by the two sets: 0")
pd.Series(station_overlap(split_table, columns)).round(3)

Events shared by the two sets: 0


validation_stations_seen_in_training     0.808
validation_records_from_seen_stations    0.964
dtype: float64

In [10]:
per_label = split_table.groupby([SPLIT_COLUMN, columns.label]).size().unstack(columns.label).rename(columns=label_names)
per_label

label,good,bad
split,,
train,51367,28826
validation,12846,7210


In [11]:
COMPARED_COLUMNS = ["preferred_magnitude", "path_hyp_dist_km", "origin_year"]
with_split = metadata.join(split_table[SPLIT_COLUMN])
comparison = numeric_summary_by_label(with_split, COMPARED_COLUMNS, SPLIT_COLUMN)
comparison.to_csv(split.tables_dir / "split_variables.csv")
comparison.round(2)

count     mean    std     min     25%  \
column              split                                                 
preferred_magnitude train       80193.0     4.43   0.65    1.69     4.0   
                    validation  20056.0     4.48   0.78    1.69     4.0   
path_hyp_dist_km    train       79606.0   139.65  81.88    1.04   76.39   
                    validation  19935.0   137.39  77.17    2.49   76.48   
origin_year         train       80193.0  2017.01   5.92  1967.0  2014.0   
                    validation  20056.0  2016.24   6.56  1969.0  2014.0   

                                   50%     75%     max  
column              split                               
preferred_magnitude train          4.3     4.7     8.4  
                    validation     4.3     4.7     8.0  
path_hyp_dist_km    train       137.29  190.84  867.89  
                    validation  136.63  188.84  724.67  
origin_year         train       2017.0  2021.0  2026.0  
                    validation  2017.0  2021.0  2026.0

In [12]:
bad_rate_by_year = (
    with_split.assign(year=with_split[ORIGIN_TIME_COLUMN].dt.year)
    .groupby(["year", SPLIT_COLUMN])[columns.label]
    .mean()
    .unstack(SPLIT_COLUMN)
)
bad_rate_by_year.tail(10).round(3)

split,train,validation
year,,
2017,0.330,0.331
2018,0.441,0.459
2019,0.505,0.459
2020,0.493,0.467
2021,0.472,0.500
2022,0.399,0.353
2023,0.389,0.383
2024,0.433,0.432
2025,0.376,0.399
